# Phase 2 L-NER - Architecture 1 (LLaMA-style decoder) + LoRMA

**Phase-1 model:** Arch 1 decoder.  **Default adaptation:** **LoRMA+** (low-rank *multiplicative* update, the compulsory method). All other methods are one `METHOD = ...` switch away.  **Head:** BiGRU over the decoder states (adds right context).

**This notebook is self-contained** - run the cells top to bottom. You normally only edit the *Configuration* cell.

| step | what happens |
|---|---|
| 1-3 | setup, configuration, your Phase-1 tokenizer |
| 4-6 | load data, tokens with exact character offsets, windows and class balance |
| 7 | decoding, strict metrics, pipeline self-test |
| 8-10 | load your Phase-1 model, adaptation methods (lorma, lora, adapter, bitfit, partial, full), NER model and training code |
| 11-12 | run: train, validate, predict test, write `submission.zip` |

*Cells labelled `# helper` are collapsed plumbing (parsing, decoding, your unchanged Phase-1 code); you never need to open them.*

**Before running:** attach your Phase-1 checkpoint dataset and `kratikasum/tokenizer` as Kaggle inputs and keep the secret `HF`. Run once with `SMOKE_TEST = True`, then switch it off. Only Phase-1 weights are used.

**Outputs** (in `/kaggle/working/<RUN_NAME>/`): `prediction.json`, `submission.zip` (`{doc_id: [[start, end, label], ...]}`), `results.json`, saved weights.

## 1. Setup

In [44]:
import os, re, sys, json, glob, math, time, random, zipfile, gc, pickle, copy
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict, fields
from typing import List, Dict, Tuple, Optional
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import login, hf_hub_download
from tqdm.auto import tqdm

try:                                                     # Kaggle secret 'HF' (same as in your Phase-1 notebooks)
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
except Exception as ex:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    print(f"Kaggle secret 'HF' not available ({type(ex).__name__}); using env HF_TOKEN / local files")

print('CUDA available:', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))
N_GPU = torch.cuda.device_count()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
use_bf16 = device == 'cuda' and torch.cuda.get_device_capability()[0] >= 8      # T4/P100: fp16 + GradScaler
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


CUDA available: True
GPU 0: Tesla T4
GPU 1: Tesla T4


## 2. Configuration
Every setting of the experiment is here.

In [45]:
# ---------- where things are ----------
HF_DATASET_REPO = 'Exploration-Lab/CS-781-Capstone'
PHASE2_DIR      = 'Phase_2_LNER'        # folder with the task data inside that dataset
LOCAL_DATA_DIR  = None                  # or e.g. '/kaggle/input/<dataset>/Phase_2_LNER'
PHASE1_CKPT     = ''                    # your Phase-1 best.pt / last.pt   ('' = search /kaggle/input)
CKPT_HINT       = 'arch1'            # used by that search: path must contain this text
RUN_NAME        = 'p2_arch1_lorma'
OUT_ROOT        = '/kaggle/working' if os.path.isdir('/kaggle/working') else './outputs'

# ---------- data handling ----------
SEED, VAL_FRAC  = 42, 0.12              # validation = 12% of documents (0 = train on everything)
PRETOK          = 'punct'               # how text is cut before BPE: 'punct' (words + punctuation) or 'ws' (whitespace words)
WINDOW, STRIDE  = 2048, 1024              # context window and step, in tokens
NEG_KEEP        = 0.6                   # share of entity-free (boilerplate) windows used per epoch
CW_POWER        = 0.3                   # rare-label loss weight = (count_O / count_label) ** CW_POWER   (0 = off)

# ---------- adaptation method ----------
METHOD = 'lorma'                        # 'lorma' | 'lora' | 'adapter' | 'bitfit' | 'partial' | 'full'   (see step 9)
HEAD   = 'bigru'                        # 'bigru' (adds right context to a causal model) or 'linear'
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 64, 0.05   # 'lora' / 'lorma': rank, scale, dropout
PEFT_TARGETS = ('q', 'k', 'v', 'o', 'gate', 'up', 'down')   # linears that get LoRA / LoRMA matrices
ADAPTER_DIM = 64                        # 'adapter': bottleneck width
N_UNFREEZE = 4                          # 'partial': how many top blocks are trained
RESUME_TRAINABLE = ''                   # path of a saved *_trainable.pt = skip training

# ---------- training ----------
EPOCHS, LR, HEAD_LR = 25, 3e-4, 1e-3     # LR = adapted backbone parameters. Typical: lora/lorma 3e-4, adapter 1e-3, bitfit 1e-3, partial/full 1e-4
BATCH_SIZE, GRAD_ACCUM = 8, 2           # windows per step x accumulation
PATIENCE = 5                            # stop after 3 epochs without validation improvement
GRAD_CKPT = False                       # True = less GPU memory, ~30% slower

# ---------- quick dry run ----------
SMOKE_TEST = False                      # True = 12 documents, 10 steps: checks the whole pipeline in a few minutes
MAX_DOCS = MAX_TEST_DOCS = MAX_STEPS = None
if SMOKE_TEST:
    MAX_DOCS, MAX_TEST_DOCS, MAX_STEPS, VAL_FRAC = 12, 3, 10, 0.25
    RUN_NAME += '_smoke'
    EPOCHS = 1

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUTPUT_DIR = os.path.join(OUT_ROOT, RUN_NAME)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Your Phase-1 tokenizer
The class is your Phase-1 code, unchanged.

In [46]:
from typing import List, Dict, Tuple
class ByteLevelBPE:
    """Your tokenizer class (training code removed: we load the already-trained pickle)."""
    def __init__(self):
        self.vocab: Dict[bytes, int] = {}
        self.inverse_vocab: Dict[int, bytes] = {}
        self.merges: Dict[Tuple[bytes, bytes], Tuple[bytes, int]] = {}
        self.word_end_marker = b'</w>'
        self.reserved_tokens = [b'<pad>', b'<unk>', b'<bos>', b'<eos>']

    def tokenize(self, text: str) -> List[bytes]:
        words = text.split()
        tokens = []
        for w in words:
            word_bytes = w.encode('utf-8')
            split = [bytes([b]) for b in word_bytes] + [self.word_end_marker]
            while True:
                best_pair, best_rank = None, float('inf')
                for i in range(len(split) - 1):
                    pair = (split[i], split[i+1])
                    if pair in self.merges:
                        rank = self.merges[pair][1]
                        if rank < best_rank:
                            best_rank, best_pair = rank, pair
                if best_pair is None:
                    break
                new_split, i = [], 0
                while i < len(split):
                    if i < len(split) - 1 and split[i] == best_pair[0] and split[i+1] == best_pair[1]:
                        new_split.append(self.merges[best_pair][0])
                        i += 2
                    else:
                        new_split.append(split[i])
                        i += 1
                split = new_split
            tokens.extend(split)
        return tokens

    def encode(self, text: str) -> List[int]:
        unk_id = self.vocab[b'<unk>']
        return [self.vocab.get(token, unk_id) for token in self.tokenize(text)]

    def decode(self, token_ids: List[int]) -> str:
        skip_ids = {self.vocab[t] for t in self.reserved_tokens if t in self.vocab}
        bytes_list = []
        for tid in token_ids:
            if tid in skip_ids:
                continue
            bytes_list.append(self.inverse_vocab.get(tid, b''))
        joined = b''.join(bytes_list)
        return joined.replace(self.word_end_marker, b' ').decode('utf-8', errors='replace').strip()

    @classmethod
    def load(cls, filepath: str):
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
        bpe = cls()
        bpe.vocab = data['vocab']
        bpe.inverse_vocab = {v: k for k, v in bpe.vocab.items()}
        bpe.merges = data['merges']
        return bpe

    def __len__(self):
        return len(self.vocab)

    @property
    def eos_token_id(self):
        return self.vocab.get(b'<eos>')

In [47]:
cands = glob.glob('/kaggle/input/**/bpe_tokenized_txt.pkl', recursive=True)
if cands:
    tok_path = cands[0]
else:
    import kagglehub
    tok_path = os.path.join(kagglehub.dataset_download('kratikasum/tokenizer'), 'bpe_tokenized_txt.pkl')
tokenizer = ByteLevelBPE.load(tok_path)
VOCAB_SIZE = max(tokenizer.vocab.values()) + 1
print(f'tokenizer: {len(tokenizer):,} entries | embedding size {VOCAB_SIZE:,} | marker {tokenizer.word_end_marker}')

tokenizer: 48,000 entries | embedding size 48,000 | marker b'</w>'


## 4. Load the data
Two training folds (documents + gold entities) and the unlabeled test fold. The gold offsets are **checked against the entity text** so an off-by-one convention can't slip through.

In [48]:
# helper: file lookup + tolerant JSON parsing
def find_data_file(name, sub):
    """Local override -> attached Kaggle dataset -> Hugging Face dataset repo."""
    if LOCAL_DATA_DIR:
        for p in (os.path.join(LOCAL_DATA_DIR, sub, name), os.path.join(LOCAL_DATA_DIR, name)):
            if os.path.exists(p):
                return p
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if hits:
        return hits[0]
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=f'{PHASE2_DIR}/{sub}/{name}',
                           repo_type='dataset', token=HF_TOKEN)


def load_texts(path):
    raw = json.load(open(path, encoding='utf-8'))
    if isinstance(raw, list):
        raw = {str(i): r for i, r in enumerate(raw)}
    return {str(k): (v.get('text', '') if isinstance(v, dict) else v) for k, v in raw.items()}


def _first(d, keys):
    return next((d[k] for k in keys if k in d), None)


def parse_entities(val):
    """-> [(start, end, label, text_or_None)]. Accepts [s, e, label(, text)] rows or {'start':..,'end':..,'label':..} dicts."""
    if isinstance(val, dict):
        val = _first(val, ['entities', 'annotations', 'labels', 'spans']) or []
    ents = []
    for e in val:
        if isinstance(e, (list, tuple)):
            ents.append((int(e[0]), int(e[1]), str(e[2]), e[3] if len(e) > 3 else None))
        elif isinstance(e, dict):
            ents.append((int(_first(e, ['start', 'start_char', 'start_offset', 'begin'])),
                         int(_first(e, ['end', 'end_char', 'end_offset', 'stop'])),
                         str(_first(e, ['label', 'type', 'entity', 'tag'])),
                         _first(e, ['text', 'entity_text', 'span'])))
    return ents


def detect_end_convention(texts, gold_raw):
    """How often is text[s:e] (exclusive end) vs text[s:e+1] (inclusive end) equal to the supplied entity text?"""
    excl = incl = tot = 0
    for d, ents in gold_raw.items():
        t = texts.get(d, '')
        for s, e, _, txt in ents:
            if txt is not None:
                tot += 1; excl += (t[s:e] == txt); incl += (t[s:e + 1] == txt)
    return excl, incl, tot


def load_labels():
    try:
        labs = []
        for l in open(find_data_file('labels.txt', 'Train'), encoding='utf-8').read().splitlines():
            l = re.sub(r'^[BI]-', '', l.strip())
            if l and l != 'O' and l not in labs:
                labs.append(l)
        if labs:
            return labs
    except Exception as ex:
        print('labels.txt not found -> using the 12 labels of the task description:', type(ex).__name__)
    return ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']

### 4a. Labels
Each label gets a `B-` and an `I-` tag plus one shared `O` tag.

In [49]:
LABELS = load_labels()
LAB2ID = {l: i for i, l in enumerate(LABELS)}          # tag ids: O = 0, B_k = 1 + 2k, I_k = 2 + 2k
LAB_LOOKUP = {l.upper(): l for l in LABELS}
NUM_CLASSES = 1 + 2 * len(LABELS)
print(len(LABELS), 'labels', LABELS, '->', NUM_CLASSES, 'tags')

12 labels ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO'] -> 25 tags


### 4b. Training folds

In [50]:
texts, gold_raw = {}, {}
for fold in ('fold1', 'fold2'):
    t = load_texts(find_data_file(f'{fold}-text.json', 'Train'))
    g = json.load(open(find_data_file(f'{fold}-gold.json', 'Train'), encoding='utf-8'))
    assert not (set(t) & set(texts)), 'document ids repeated across folds'
    texts.update(t)
    gold_raw.update({str(k): parse_entities(v) for k, v in g.items()})
print(f'{len(texts)} documents | {sum(len(v) for v in gold_raw.values())} gold entities')

70 documents | 8393 gold entities


### 4c. Offset check and clean-up
The task example `[18, 40, "COURT"]` means the end offset is exclusive. We confirm it on the data (and convert if it is not).

In [51]:
excl, incl, tot = detect_end_convention(texts, gold_raw)
END_INCLUSIVE = incl > excl
print(f'text[s:e] == entity_text: {excl}/{tot} | text[s:e+1] == entity_text: {incl}/{tot}  ->  end is',
      'INCLUSIVE (will convert)' if END_INCLUSIVE else 'exclusive (as expected)')

gold, bad = {}, Counter()
for d, text in texts.items():
    keep = {}
    for s, e, lab, _ in gold_raw.get(d, []):
        e = e + 1 if END_INCLUSIVE else e
        lab = LAB_LOOKUP.get(re.sub(r'^[BI]-', '', lab.strip()).upper())
        if lab is None:                  bad['unknown label'] += 1
        elif not (0 <= s < e <= len(text)): bad['bad range'] += 1
        elif (s, e, lab) in keep:        bad['duplicate'] += 1
        else:                            keep[(s, e, lab)] = 1
    gold[d] = sorted(keep)
print('dropped while cleaning:', dict(bad) or 'nothing')
print('entities per label:', {l: c for l, c in Counter(x[2] for v in gold.values() for x in v).items()})

n_over = 0                               # overlapping gold spans cannot be expressed as BIO tags
for v in gold.values():
    run_end = -1
    for s, e, _ in v:
        n_over += s < run_end; run_end = max(run_end, e)
print('overlapping / nested gold spans:', n_over, '(the longer span is kept)')

text[s:e] == entity_text: 8393/8393 | text[s:e+1] == entity_text: 17/8393  ->  end is exclusive (as expected)
dropped while cleaning: nothing
entities per label: {'RESP': 381, 'DATE': 1575, 'COURT': 1013, 'WIT': 187, 'APP': 407, 'STAT': 1260, 'A.COUNSEL': 195, 'PREC': 1183, 'JUDGE': 245, 'R.COUNSEL': 184, 'CASENO': 760, 'AUTH': 1003}
overlapping / nested gold spans: 0 (the longer span is kept)


### 4d. Test fold

In [52]:
test_texts = load_texts(find_data_file('fold3-text.json', 'Test'))
if MAX_DOCS:                             # only active in SMOKE_TEST mode
    keep_ids = sorted(texts)[:MAX_DOCS]
    texts, gold = {k: texts[k] for k in keep_ids}, {k: gold[k] for k in keep_ids}
if MAX_TEST_DOCS:
    test_texts = {k: test_texts[k] for k in sorted(test_texts)[:MAX_TEST_DOCS]}
print(len(texts), 'train documents |', len(test_texts), 'test documents')

70 train documents | 35 test documents


## 5. Tokens **with character offsets**
Your Phase-1 tokenizer splits on whitespace and forgets where each word was. The strict metric needs exact character boundaries, so each text is first cut into *pieces* (words and single punctuation marks), every piece is BPE-encoded with **your** tokenizer, and each token's character span is recovered from its byte length. The `</w>` marker has no characters, so a token that is only `</w>` has zero width and is ignored.

In [53]:
# helper: tokenizer wrapper that keeps character offsets
class OffsetEncoder:
    def __init__(self, tok, mode='punct'):
        self.tok = tok
        self.rx = re.compile(r'\w+|[^\w\s]') if mode == 'punct' else re.compile(r'\S+')   # 'ws' = Phase-1 whitespace words
        self.cache, self.n_bad = {}, 0
        self.unk, self.marker = tok.vocab[b'<unk>'], tok.word_end_marker

    def _piece(self, piece):
        if piece in self.cache:
            return self.cache[piece]
        toks = self.tok.tokenize(piece)
        ids = [self.tok.vocab.get(t, self.unk) for t in toks]
        c_of_b = [ci for ci, ch in enumerate(piece) for _ in ch.encode('utf-8')]          # byte index -> char index
        spans, pos = [], 0
        for t in toks:
            body = len(t) - len(self.marker) if t.endswith(self.marker) else len(t)
            if body <= 0 or not c_of_b:
                spans.append((len(piece), len(piece)))                                       # zero width
            else:
                spans.append((c_of_b[min(pos, len(c_of_b) - 1)], c_of_b[min(pos + body - 1, len(c_of_b) - 1)] + 1))
            pos += max(body, 0)
        if pos != len(c_of_b):                                                               # cannot happen with byte-level BPE
            self.n_bad += 1
            spans = [(0, len(piece))] * len(toks)
        self.cache[piece] = (ids, spans)
        return ids, spans

    def encode(self, text):
        ids, st, en = [], [], []
        for m in self.rx.finditer(text):
            i, sp = self._piece(m.group())
            ids += i
            st += [m.start() + a for a, _ in sp]
            en += [m.start() + b for _, b in sp]
        return np.asarray(ids, np.int32), np.asarray(st, np.int32), np.asarray(en, np.int32)

In [54]:
# helper: gold character spans -> one BIO tag per token
def token_labels(starts, ends, ents, lab2id):
    """Tag id per token (-100 for zero-width tokens). On overlap the longer span wins."""
    T = len(starts)
    y = np.zeros(T, np.int64)
    zw = starts == ends
    y[zw] = -100
    stats = Counter()
    for s, e, lab in sorted(ents, key=lambda x: (-(x[1] - x[0]), x[0])):
        i = int(np.searchsorted(starts, s, 'left'))            # first token starting at / after s
        j = int(np.searchsorted(ends, e, 'right')) - 1         # last token ending at / before e
        while i <= j and i < T and zw[i]: i += 1
        while j >= i and zw[j]: j -= 1
        if i > j or i >= T or j < 0:
            stats['no_token'] += 1; continue
        stats['exact' if (starts[i] == s and ends[j] == e) else 'inexact'] += 1
        if (y[i:j + 1] > 0).any():
            stats['overlap_dropped'] += 1; continue
        y[i] = 1 + 2 * lab2id[lab]
        for t in range(i + 1, j + 1):
            if not zw[t]:
                y[t] = 2 + 2 * lab2id[lab]
    return y, stats


def boundary_ceiling(texts_, gold_, mode, n_docs=40):
    """Share of gold spans whose start AND end fall on token boundaries (= best possible strict recall)."""
    enc, ok, tot = OffsetEncoder(tokenizer, mode), 0, 0
    for d in list(texts_)[:n_docs]:
        _, st, en = enc.encode(texts_[d])
        S, E = set(st.tolist()), set(en.tolist())
        for s, e, _ in gold_[d]:
            tot += 1; ok += (s in S and e in E)
    return ok / max(1, tot), tot

### 5a. How well can this tokenisation hit exact boundaries?

In [55]:
for mode in ('punct', 'ws'):
    frac, tot = boundary_ceiling(texts, gold, mode)
    print(f'pre-tokenisation {mode!r:8}: {100 * frac:6.2f}% of {tot} gold spans are recoverable exactly')
print('-> using PRETOK =', repr(PRETOK))

pre-tokenisation 'punct' :  99.38% of 4514 gold spans are recoverable exactly
pre-tokenisation 'ws'    :  66.02% of 4514 gold spans are recoverable exactly
-> using PRETOK = 'punct'


### 5b. Tokenise every document

In [56]:
encoder = OffsetEncoder(tokenizer, PRETOK)
PAD_ID = tokenizer.vocab[b'<pad>']


def build_docs(texts_, gold_=None):
    docs, stats = [], Counter()
    for d, t in texts_.items():
        ids, st, en = encoder.encode(t)
        doc = dict(id=d, text=t, ids=ids, starts=st, ends=en, ents=gold_[d] if gold_ else None, y=None)
        if gold_:
            doc['y'], s = token_labels(st, en, gold_[d], LAB2ID)
            stats.update(s)
        docs.append(doc)
    return docs, stats

all_docs, align_stats = build_docs(texts, gold)
test_docs, _ = build_docs(test_texts)
lens = np.array([len(d['ids']) for d in all_docs])
print(f'{len(all_docs)} train docs, tokens per doc: mean {lens.mean():.0f} / p95 {np.percentile(lens, 95):.0f} / max {lens.max()}')
print('gold -> tags:', dict(align_stats), '| byte fallbacks:', encoder.n_bad)

70 train docs, tokens per doc: mean 6800 / p95 13005 / max 24189
gold -> tags: {'exact': 8333, 'inexact': 50, 'no_token': 10} | byte fallbacks: 0


## 6. Validation split, context windows, class balance
* validation = whole **documents** held out;
* documents are cut into overlapping windows of `WINDOW` tokens (a window is the model's context);
* windows with no entity (legal boilerplate) are down-sampled to `NEG_KEEP` each epoch and rare tags get a larger loss weight.

In [57]:
order = np.random.RandomState(SEED).permutation(len(all_docs))
n_val = int(round(VAL_FRAC * len(all_docs)))
val_docs = [all_docs[i] for i in order[:n_val]]
train_docs = [all_docs[i] for i in order[n_val:]]
print(f'train documents {len(train_docs)} | validation documents {len(val_docs)}')

train documents 62 | validation documents 8


In [58]:
# helper: windows and batches
def window_starts(n, W, S):
    if n <= W:
        return [0]
    st = list(range(0, n - W + 1, S))
    return st + [n - W] if st[-1] + W < n else st


def make_train_items(docs):
    """-> (windows containing entities, windows without); a window is (doc_index, start_token)."""
    pos, neg = [], []
    for di, d in enumerate(docs):
        for s in window_starts(len(d['ids']), WINDOW, STRIDE):
            (pos if (d['y'][s:s + WINDOW] > 0).any() else neg).append((di, s))
    return pos, neg


def make_batch(docs, items):
    """-> ids, labels (-100 = ignore), mask   (right padded, on `device`)"""
    seqs = [(docs[di]['ids'][s:s + WINDOW], None if docs[di]['y'] is None else docs[di]['y'][s:s + WINDOW]) for di, s in items]
    L = max(len(a) for a, _ in seqs)
    ids = np.full((len(seqs), L), PAD_ID, np.int64)
    lab = np.full((len(seqs), L), -100, np.int64)
    mask = np.zeros((len(seqs), L), bool)
    for i, (a, y) in enumerate(seqs):
        ids[i, :len(a)], mask[i, :len(a)] = a, True
        if y is not None:
            lab[i, :len(y)] = y
    return torch.from_numpy(ids).to(device), torch.from_numpy(lab).to(device), torch.from_numpy(mask).to(device)

In [59]:
pos_items, neg_items = make_train_items(train_docs)
print(f'{len(pos_items)} windows with entities | {len(neg_items)} without (only {NEG_KEEP:.0%} used per epoch)')

counts = np.zeros(NUM_CLASSES)
for d in train_docs:
    counts += np.bincount(d['y'][d['y'] >= 0], minlength=NUM_CLASSES)
cw = np.ones(NUM_CLASSES, np.float32)                  # weight = (count_O / count_tag) ** CW_POWER, capped at 10
if CW_POWER > 0:
    cw[1:] = [min(10.0, (counts[0] / c) ** CW_POWER) if c > 0 else 1.0 for c in counts[1:]]
CLASS_W = torch.tensor(cw, device=device)
print(f'{100 * counts[0] / counts.sum():.1f}% of tokens are O | loss weight of each label (B, I):',
      {LABELS[k]: (round(float(cw[1 + 2 * k]), 1), round(float(cw[2 + 2 * k]), 1)) for k in range(len(LABELS))})

390 windows with entities | 1 without (only 60% used per epoch)
90.8% of tokens are O | loss weight of each label (B, I): {'APP': (8.0, 5.8), 'RESP': (8.1, 6.1), 'A.COUNSEL': (10.0, 7.6), 'R.COUNSEL': (10.0, 7.5), 'JUDGE': (9.4, 6.0), 'WIT': (10.0, 8.3), 'AUTH': (6.1, 5.7), 'COURT': (6.1, 5.5), 'STAT': (5.7, 4.2), 'PREC': (5.8, 2.9), 'DATE': (5.4, 3.6), 'CASENO': (6.7, 3.6)}


## 7. Decoding, strict metrics, error analysis
* **greedy** decoding = best tag per token; **viterbi** = best tag *path* that obeys BIO rules (no `O -> I-x`, no `B-x -> I-y`), which removes broken spans;
* each span gets a confidence (mean token probability) so low-confidence spans can be dropped;
* a prediction is correct only if `(start, end, label)` all match exactly; the macro scores average over labels.

In [60]:
# helper: tags -> character spans
def build_transitions(n_labels):
    C = 1 + 2 * n_labels
    NEG = -1e9
    trans = np.zeros((C, C), np.float32)                      # trans[prev, cur]
    start = np.zeros(C, np.float32)
    for cur in range(2, C, 2):                                # I-k may only follow B-k / I-k
        k = (cur - 2) // 2
        for prev in range(C):
            if prev not in (1 + 2 * k, 2 + 2 * k):
                trans[prev, cur] = NEG
        start[cur] = NEG
    return trans, start

TRANS, START = build_transitions(len(LABELS))


def viterbi(em):
    n, C = em.shape
    score = em[0] + START
    back = np.zeros((n, C), np.int16)
    for t in range(1, n):
        cand = score[:, None] + TRANS
        back[t] = cand.argmax(0)
        score = cand.max(0) + em[t]
    best = int(score.argmax())
    path = [best]
    for t in range(n - 1, 0, -1):
        best = int(back[t, best])
        path.append(best)
    return np.asarray(path[::-1])


def tags_to_spans(tags):
    spans, i, n = [], 0, len(tags)
    while i < n:
        t = tags[i]
        if t == 0:
            i += 1; continue
        k = (t - 1) // 2
        j = i + 1
        while j < n and tags[j] == 2 + 2 * k:
            j += 1
        spans.append((i, j - 1, k))
        i = j
    return spans


def decode_doc(doc, lp, mode='viterbi'):
    """lp: (n_tokens, C) log-probs -> [(start_char, end_char, label, confidence)] (end exclusive)."""
    st, en = doc['starts'], doc['ends']
    valid = np.nonzero(st < en)[0]                            # skip zero-width tokens
    if len(valid) == 0:
        return []
    em = lp[valid]
    tags = viterbi(em) if mode == 'viterbi' else em.argmax(1)
    prob = np.exp(em[np.arange(len(valid)), tags])
    return [(int(st[valid[a]]), int(en[valid[b]]), LABELS[k], float(prob[a:b + 1].mean()))
            for a, b, k in tags_to_spans(tags)]

In [61]:
# helper: strict precision / recall / F1
def strict_scores(gold_, pred_, labels=None):
    labels = labels or LABELS
    tp, fp, fn = Counter(), Counter(), Counter()
    for d, G in gold_.items():
        G = set(tuple(x[:3]) for x in G)
        P = set(tuple(x[:3]) for x in pred_.get(d, []))
        for x in G & P: tp[x[2]] += 1
        for x in P - G: fp[x[2]] += 1
        for x in G - P: fn[x[2]] += 1
    rows = {}
    for l in labels:
        p = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        r = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        rows[l] = dict(p=p, r=r, f1=(2 * p * r / (p + r) if p + r else 0.0), tp=tp[l], fp=fp[l], fn=fn[l], support=tp[l] + fn[l])
    act = [l for l in labels if rows[l]['support'] > 0]
    mean = lambda k, ls: float(np.mean([rows[l][k] for l in ls])) if ls else 0.0
    TP, FP, FN = sum(tp.values()), sum(fp.values()), sum(fn.values())
    mp, mr = TP / max(1, TP + FP), TP / max(1, TP + FN)
    return dict(per_label=rows, macro_p=mean('p', act), macro_r=mean('r', act), macro_f1=mean('f1', act),
                macro_f1_all_labels=mean('f1', labels), micro_f1=(2 * mp * mr / (mp + mr) if mp + mr else 0.0))


def print_scores(sc, title=''):
    print(f'--- {title} strict macro-P {sc["macro_p"]:.4f} | macro-R {sc["macro_r"]:.4f} | macro-F1 {sc["macro_f1"]:.4f} | micro-F1 {sc["micro_f1"]:.4f}')
    for l, r in sc['per_label'].items():
        print(f'   {l:10s} P {r["p"]:.3f} R {r["r"]:.3f} F1 {r["f1"]:.3f}  (support {r["support"]:5d}, tp {r["tp"]:5d}, fp {r["fp"]:5d}, fn {r["fn"]:5d})')

In [62]:
# helper: error analysis (boundary errors, label confusions, spurious spans)
def error_analysis(docs_by_id, gold_, pred_, n_examples=8):
    """Splits every wrong prediction / miss into boundary errors, label confusions and spurious / missed spans."""
    cat, conf, shifts, fp_txt, ex = Counter(), Counter(), Counter(), Counter(), []
    ov = lambda a, b: a[0] < b[1] and b[0] < a[1]
    for d, G in gold_.items():
        Gs = set(tuple(x[:3]) for x in G)
        Ps = set(tuple(x[:3]) for x in pred_.get(d, []))
        text = docs_by_id[d]['text']
        for p in Ps - Gs:
            o = [g for g in Gs if ov(g, p)]
            same = [g for g in o if g[2] == p[2]]
            if same:
                g = max(same, key=lambda g: min(g[1], p[1]) - max(g[0], p[0]))
                cat['boundary error (FP side)'] += 1
                shifts[(max(-3, min(3, p[0] - g[0])), max(-3, min(3, p[1] - g[1])))] += 1
                if len(ex) < n_examples:
                    ex.append((p[2], text[g[0]:g[1]], text[p[0]:p[1]]))
            elif o:
                cat['label confusion (FP side)'] += 1
                g = max(o, key=lambda g: min(g[1], p[1]) - max(g[0], p[0]))
                conf[(g[2], p[2])] += 1
            else:
                cat['spurious span (no gold overlap)'] += 1
                fp_txt[(p[2], text[p[0]:p[1]].lower()[:40])] += 1
        for g in Gs - Ps:
            o = [p for p in Ps if ov(g, p)]
            if any(p[2] == g[2] for p in o):
                cat['boundary error (FN side)'] += 1
            elif o:
                cat['label confusion (FN side)'] += 1
            else:
                cat['missed entirely (FN)'] += 1
    print('error categories:'); [print(f'   {k:36s} {v}') for k, v in cat.most_common()]
    print('most common label confusions (gold -> predicted):', conf.most_common(8))
    print('boundary shifts (start_delta, end_delta) in characters, clipped to +-3:', shifts.most_common(8))
    print('boundary-error examples (label | gold text | predicted text):'); [print('   ', e) for e in ex]
    print('most frequent spurious predictions (boilerplate suspects):', fp_txt.most_common(10))
    return dict(categories=dict(cat), confusions={f'{a}->{b}': c for (a, b), c in conf.items()},
                boundary_shifts={f'{a},{b}': c for (a, b), c in shifts.items()})

### 7a. Pipeline self-test (no model involved)
Gold spans -> tags -> perfect log-probabilities -> decoder -> character spans. This must reproduce the gold spans except for the tokenisation ceiling printed in step 5a. If it does, offsets, tagging and decoding are consistent.

In [63]:
def oracle_decode_check(docs, mode='viterbi'):
    gold_, pred_ = {}, {}
    for d in docs:
        y = np.maximum(d['y'], 0)
        lp = np.full((len(y), NUM_CLASSES), -20.0, np.float32)
        lp[np.arange(len(y)), y] = 0.0
        pred_[d['id']], gold_[d['id']] = decode_doc(d, lp, mode), d['ents']
    return strict_scores(gold_, pred_)

for m in ('greedy', 'viterbi'):
    sc = oracle_decode_check(all_docs[:50], m)
    print(f'oracle [{m}]: strict macro-F1 {sc["macro_f1"]:.4f} | micro-F1 {sc["micro_f1"]:.4f}   (upper bound for this tokenisation)')

oracle [greedy]: strict macro-F1 0.9941 | micro-F1 0.9927   (upper bound for this tokenisation)
oracle [viterbi]: strict macro-F1 0.9941 | micro-F1 0.9927   (upper bound for this tokenisation)


## 8. Load your Phase-1 model
The architecture code is copied unchanged from your Phase-1 notebook (collapsed); the weights come from your checkpoint.

In [64]:
# helper: Phase-1 model code, unchanged
import math
from dataclasses import dataclass
from typing import List, Optional
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

@dataclass
class Config:
    vocab_size: int = 32000
    d_model: int = 768
    n_layer: int = 12
    n_head: int = 12
    n_kv_head: Optional[int] = None      # None -> MHA; < n_head -> GQA
    max_seq_len: int = 2048
    rope_theta: float = 10000.0
    rope_pct: float = 1.0                # fraction of the head dim that is rotated (partial RoPE)
    dropout: float = 0.0
    tie_embeddings: bool = False
    qk_norm: bool = False
    grad_ckpt: bool = False
    value_residual: bool = False         # later layers mix in the first layer's values
    mtp_heads: int = 0                   # extra multi-token-prediction heads (t+2, t+3, ...)
    mtp_weight: float = 0.3
    n_sources: int = 0                   # >0 -> source embedding (SC/Acts/HC/Tribunal) + 1 "unknown"
    src_drop: float = 0.3                # prob. of replacing the source id by "unknown" in training

    @property
    def head_dim(self):
        return self.d_model // self.n_head

    @property
    def kv_heads(self):
        return self.n_kv_head or self.n_head

    @property
    def ff_dim(self):                    # SwiGLU hidden size, parameter-matched to a 4*d GELU MLP
        return int(round(8 / 3 * self.d_model / 64)) * 64

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.w = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        xf = x.float()
        out = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (out * self.w.float()).type_as(x)


class Rotary(nn.Module):
    def __init__(self, dim, max_len, theta):
        super().__init__()
        inv = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / max(dim, 1)))
        freqs = torch.outer(torch.arange(max_len).float(), inv)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, T, start=0):
        """cos/sin for positions start .. start+T-1 (start > 0 only while generating with a cache)."""
        return self.cos[start:start + T], self.sin[start:start + T]


def apply_rope(x, cos, sin, rot_dim):
    """x: (B,H,T,hd). Rotates the first rot_dim dims (half-split convention)."""
    if rot_dim == 0:
        return x
    xr, xp = x[..., :rot_dim], x[..., rot_dim:]
    x1, x2 = xr.float().chunk(2, dim=-1)
    c, s = cos[None, None], sin[None, None]
    xr = torch.cat([x1 * c - x2 * s, x1 * s + x2 * c], dim=-1).to(x.dtype)
    return torch.cat([xr, xp], dim=-1)


class MLP(nn.Module):
    """SwiGLU feed-forward."""
    def __init__(self, cfg):
        super().__init__()
        d, f = cfg.d_model, cfg.ff_dim
        self.gate = nn.Linear(d, f, bias=False)
        self.up = nn.Linear(d, f, bias=False)
        self.down = nn.Linear(f, d, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

# --------------------------------------------------------------------------------------
# KV cache + attention
# --------------------------------------------------------------------------------------
class KVCache:
    """Pre-allocated key/value buffer for ONE attention layer (stored un-expanded, i.e. Hkv heads).
    Lets generation process one new token per step instead of re-running the whole prefix."""

    def __init__(self, max_len):
        self.max_len, self.n, self.k, self.v = max_len, 0, None, None

    def append(self, k, v):
        if self.k is None:
            B, H, _, hd = k.shape
            self.k = k.new_empty(B, H, self.max_len, hd)
            self.v = v.new_empty(B, H, self.max_len, hd)
        T = k.size(2)
        self.k[:, :, self.n:self.n + T] = k
        self.v[:, :, self.n:self.n + T] = v
        self.n += T

    def get(self, window=None):
        s = 0 if window is None else max(0, self.n - window)
        return self.k[:, :, s:self.n], self.v[:, :, s:self.n]


def sdpa_gqa(q, k, v, mask=None, causal=False, p=0.0):
    """q: (B,H,T,hd)   k,v: (B,Hkv,S,hd).  Handles GQA without copying the cache when T == 1."""
    B, H, T, hd = q.shape
    Hkv = k.size(1)
    if T == 1 and mask is None:                       # single-token decode step
        if Hkv != H:                                  # fold the query-head group into the "length" axis
            r = H // Hkv
            out = F.scaled_dot_product_attention(q.reshape(B, Hkv, r, hd), k, v)
            return out.reshape(B, H, 1, hd)
        return F.scaled_dot_product_attention(q, k, v)
    if Hkv != H:
        r = H // Hkv
        k, v = k.repeat_interleave(r, dim=1), v.repeat_interleave(r, dim=1)
    return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=p,
                                          is_causal=causal and mask is None)

class Attention(nn.Module):
    """MHA/GQA self-attention with optional RoPE (or NoPE), QK-norm, value residual, sliding-window mask
    and KV cache.  Returns (output, v_of_the_new_tokens)."""

    def __init__(self, cfg, causal=True, use_rope=True, value_residual=False):
        super().__init__()
        d = cfg.d_model
        self.H, self.Hkv, self.hd = cfg.n_head, cfg.kv_heads, cfg.head_dim
        assert self.H % self.Hkv == 0, "n_head must be divisible by n_kv_head"
        self.q = nn.Linear(d, self.H * self.hd, bias=False)
        self.k = nn.Linear(d, self.Hkv * self.hd, bias=False)
        self.v = nn.Linear(d, self.Hkv * self.hd, bias=False)
        self.o = nn.Linear(self.H * self.hd, d, bias=False)
        self.q_norm = RMSNorm(self.hd) if cfg.qk_norm else None
        self.k_norm = RMSNorm(self.hd) if cfg.qk_norm else None
        self.p_drop = cfg.dropout
        self.causal = causal
        self.rot_dim = (int(self.hd * cfg.rope_pct) // 2 * 2) if use_rope else 0
        self.lam = nn.Parameter(torch.zeros(self.Hkv, 1, 1)) if value_residual else None

    def forward(self, x, rope=None, v_first=None, mask=None, cache=None, window=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.H, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.Hkv, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.Hkv, self.hd).transpose(1, 2)
        if self.lam is not None and v_first is not None:
            lam = torch.sigmoid(self.lam).to(v.dtype)
            v = (1 - lam) * v + lam * v_first
        v_new = v
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        if self.rot_dim:
            cos, sin = rope
            q, k = apply_rope(q, cos, sin, self.rot_dim), apply_rope(k, cos, sin, self.rot_dim)
        causal = self.causal
        if cache is not None:
            cache.append(k, v)
            k, v = cache.get(window if T == 1 else None)   # window only matters for 1-token steps
            causal = T > 1                                  # prefill: causal; decode step: sees everything cached
        out = sdpa_gqa(q, k, v, mask=mask, causal=causal, p=self.p_drop if self.training else 0.0)
        return self.o(out.transpose(1, 2).reshape(B, T, self.H * self.hd)), v_new

class LMHead(nn.Module):
    """Final norm + unembedding (optionally tied) + optional multi-token-prediction heads."""

    def __init__(self, cfg, tok_emb: nn.Embedding):
        super().__init__()
        self.cfg = cfg
        self.norm = RMSNorm(cfg.d_model)
        self.proj = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.proj.weight = tok_emb.weight
        self.mtp = nn.ModuleList([nn.Linear(cfg.d_model, cfg.d_model, bias=False)
                                  for _ in range(getattr(cfg, "mtp_heads", 0))])

    def logits(self, h):
        return self.proj(self.norm(h))

    def forward(self, h, targets=None):
        h = self.norm(h)
        logits = self.proj(h)
        if targets is None:
            return logits, None
        V = logits.size(-1)
        loss = F.cross_entropy(logits.reshape(-1, V).float(), targets.reshape(-1), ignore_index=-100)
        self.main_loss = loss.detach()          # next-token loss only (without MTP terms), for logging
        w = getattr(self.cfg, "mtp_weight", 0.3)
        for k, p in enumerate(self.mtp, start=1):
            if targets.size(1) <= k:
                break
            hk = h[:, :-k] + F.silu(p(h[:, :-k]))
            lk = self.proj(hk)
            loss = loss + w * F.cross_entropy(lk.reshape(-1, V).float(), targets[:, k:].reshape(-1),
                                              ignore_index=-100)
        return logits, loss

def ckpt_call(fn, enabled, *args):
    return checkpoint(fn, *args, use_reentrant=False) if enabled else fn(*args)


def init_weights(model, depth):
    for m in model.modules():
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)
    std = 0.02 / math.sqrt(2 * depth)
    for n, p in model.named_parameters():
        if n.endswith(("o.weight", "down.weight")):
            nn.init.normal_(p, 0.0, std)


class BaseLM(nn.Module):
    """Common interface used by training.py / generation.py:
         forward(x, targets=None, src=None) -> (logits, loss)
         prefill(prompt_ids, max_new)       -> (logits_for_next_token, state)      [generation]
         step(token_ids, state)             -> logits_for_next_token               [generation]
       `eval_offset` = number of leading label positions that are NOT scored (encoder-decoder only)."""
    eval_offset = 0

    def num_params(self, non_embedding=False):
        n = sum(p.numel() for p in self.parameters())
        return n - self.tok.weight.numel() if non_embedding else n

class Block(nn.Module):
    """Pre-norm residual block (attention + SwiGLU)."""

    def __init__(self, cfg, value_residual=False):
        super().__init__()
        self.n1, self.n2 = RMSNorm(cfg.d_model), RMSNorm(cfg.d_model)
        self.attn = Attention(cfg, value_residual=value_residual)
        self.mlp = MLP(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, rope, v_first=None, cache=None):
        h, v = self.attn(self.n1(x), rope, v_first, cache=cache)
        x = x + self.drop(h)
        return x + self.drop(self.mlp(self.n2(x))), v

class DecoderLM(BaseLM):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.tok = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.src_emb = nn.Embedding(cfg.n_sources + 1, cfg.d_model) if cfg.n_sources else None
        self.rope = Rotary(int(cfg.head_dim * cfg.rope_pct) // 2 * 2, cfg.max_seq_len, cfg.rope_theta)
        self.blocks = nn.ModuleList([Block(cfg, cfg.value_residual and i > 0) for i in range(cfg.n_layer)])
        self.head = LMHead(cfg, self.tok)
        init_weights(self, cfg.n_layer)

    def hidden(self, idx, src=None, caches=None, start=0):
        cfg = self.cfg
        B, T = idx.shape
        x = self.tok(idx)
        if self.src_emb is not None:
            unk = cfg.n_sources
            if src is None:                                  # unknown source (validation / test)
                src = torch.full((B,), unk, device=idx.device, dtype=torch.long)
            elif self.training and cfg.src_drop > 0:
                drop = torch.rand(B, device=idx.device) < cfg.src_drop
                src = torch.where(drop, torch.full_like(src, unk), src)
            x = x + self.src_emb(src)[:, None, :]
        rope = self.rope(T, start)
        v_first = None
        for i, blk in enumerate(self.blocks):
            if caches is None:
                x, v = ckpt_call(blk, cfg.grad_ckpt and self.training, x, rope, v_first)
            else:
                x, v = blk(x, rope, v_first, cache=caches[i])
            if v_first is None and cfg.value_residual:
                v_first = v
        return x

    def forward(self, idx, targets=None, src=None):
        return self.head(self.hidden(idx, src), targets)

    # ---- generation (KV cache) ----
    def prefill(self, idx, max_new):
        caches = [KVCache(idx.shape[1] + max_new) for _ in self.blocks]
        h = self.hidden(idx, caches=caches)
        return self.head.logits(h[:, -1]), caches

    def step(self, tok, caches):
        h = self.hidden(tok, caches=caches, start=caches[0].n)
        return self.head.logits(h[:, -1])

In [65]:
def find_ckpt():
    if PHASE1_CKPT and os.path.exists(PHASE1_CKPT):
        return PHASE1_CKPT
    cands = sorted(glob.glob('/kaggle/input/**/*.pt', recursive=True))
    hit = [c for c in cands if CKPT_HINT.lower() in c.lower()] or cands
    if not hit:
        raise FileNotFoundError('No Phase-1 checkpoint found: attach it as a Kaggle dataset or set PHASE1_CKPT')
    print('checkpoint candidates:', *hit, sep='\n   ')
    return ([c for c in hit if 'best' in os.path.basename(c)] or [c for c in hit if 'last' in os.path.basename(c)] or hit)[0]

ckpt_path = find_ckpt()
PHASE1 = torch.load(ckpt_path, map_location='cpu', weights_only=False)     # {'model', 'cfg', 'step', 'val_loss'}
D_MODEL = PHASE1['cfg']['d_model']
ARCH_NAME = 'arch1_decoder'
print('using', ckpt_path, '| Phase-1 step', PHASE1.get('step'), '| val loss', PHASE1.get('val_loss'))

checkpoint candidates:
   /kaggle/input/datasets/kratikasum/model-arch-1/last.pt
using /kaggle/input/datasets/kratikasum/model-arch-1/last.pt | Phase-1 step 1525 | val loss 2.419223958711014


In [66]:
# helper: rebuild the Phase-1 model from the checkpoint and expose its hidden states
def load_phase1_backbone(keep_layers=None):
    valid = {f.name for f in fields(Config)}
    cfg = Config(**{k: v for k, v in PHASE1['cfg'].items() if k in valid})
    cfg.grad_ckpt, cfg.dropout = GRAD_CKPT, 0.1
    m = DecoderLM(cfg)
    missing, unexpected = m.load_state_dict(PHASE1['model'], strict=False)
    assert not missing, f'missing keys when loading Phase-1 weights: {missing[:5]}'
    m.head.proj, m.head.mtp = nn.Identity(), nn.ModuleList()                # LM output layer / MTP heads are not needed
    return m


def backbone_features(m, ids, mask):
    """(batch, tokens) ids -> (batch, tokens, d_model). Causal model, right padding: padding never affects real tokens."""
    return m.head.norm(m.hidden(ids))


def get_blocks(m):
    return m.blocks


def get_final_norm(m):
    return m.head.norm

In [67]:
_probe = load_phase1_backbone()
print(f'{ARCH_NAME}: {sum(p.numel() for p in _probe.parameters()) / 1e6:.1f}M parameters loaded from Phase 1 (d_model {D_MODEL})')
del _probe

arch1_decoder: 143.9M parameters loaded from Phase 1 (d_model 768)


## 9. Adaptation methods (choose with `METHOD` in the configuration cell)
The Phase-1 weights are frozen first; each method then makes a different small part trainable. The NER head is always trained.

| `METHOD` | what is trained | idea |
|---|---|---|
| `'lorma'` | low-rank matrices, **multiplicative** (LoRMA+) | `W' = (I + (α/r)·B·A)·W0`, computed as `z = W0·x`, `y = z + (α/r)·B·(A·z)` |
| `'lora'` | low-rank matrices, additive | `W' = W0 + (α/r)·B·A` |
| `'adapter'` | Houlsby bottleneck adapters after the attention-output and FFN-output projections | `y' = y + Up(GELU(Down(y)))` |
| `'bitfit'` | only bias terms and norm gains | sparse fine-tuning |
| `'partial'` | the last `N_UNFREEZE` blocks and the final norm | layer-wise fine-tuning |
| `'full'` | every parameter | full fine-tuning |

`B` (LoRA / LoRMA+) and the adapter up-projection start at zero, so every method starts exactly at the Phase-1 model.

In [68]:
# helper: all adaptation methods
class LoRALinear(nn.Module):
    """mode 'lora':  y = W0 x + s·B·A·x          mode 'lorma' (LoRMA+):  y = z + s·B·A·z  with z = W0 x   (= (I + s·B·A)·W0·x)"""
    def __init__(self, base, r, alpha, mode, dropout=0.0):
        super().__init__()
        self.base, self.mode, self.scale = base, mode, alpha / r
        d_out, d_in = base.weight.shape
        self.A = nn.Parameter(torch.empty(r, d_in if mode == 'lora' else d_out))
        self.B = nn.Parameter(torch.zeros(d_out, r))              # B = 0 -> starts exactly at the Phase-1 model
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        if self.mode == 'lora':
            return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)
        z = F.linear(x, self.base.weight)                         # W0 x
        z = z + self.scale * F.linear(F.linear(z, self.A), self.B)
        return z if self.base.bias is None else z + self.base.bias


class AdaptedLinear(nn.Module):
    """Frozen linear followed by a zero-initialised residual bottleneck adapter."""
    def __init__(self, base, bottleneck):
        super().__init__()
        self.base = base
        d = base.weight.shape[0]
        self.down, self.up = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, std=0.02)
        nn.init.zeros_(self.down.bias); nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        y = self.base(x)
        return y + self.up(F.gelu(self.down(y)))


def wrap_linears(module, targets, make):
    """Replace every nn.Linear called `targets` (q, k, v, o, ...) inside `module` by make(linear)."""
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, make(child))
        else:
            wrap_linears(child, targets, make)


def apply_method(backbone, method):
    blocks = get_blocks(backbone)
    for p in backbone.parameters():
        p.requires_grad_(False)                                   # freeze Phase-1 first
    if method in ('lora', 'lorma'):
        wrap_linears(blocks, set(PEFT_TARGETS), lambda l: LoRALinear(l, LORA_R, LORA_ALPHA, method, LORA_DROPOUT))
    elif method == 'adapter':
        wrap_linears(blocks, {'o', 'down'}, lambda l: AdaptedLinear(l, ADAPTER_DIM))
    elif method == 'bitfit':
        for n, p in backbone.named_parameters():
            if n.endswith('.bias') or n.endswith('.w'):          # biases and RMSNorm gains
                p.requires_grad_(True)
    elif method == 'partial':
        for part in list(blocks)[-N_UNFREEZE:] + [get_final_norm(backbone)]:
            for p in part.parameters():
                p.requires_grad_(True)
    elif method == 'full':
        for p in backbone.parameters():
            p.requires_grad_(True)
    else:
        raise ValueError(f'unknown METHOD {method!r}')

## 10. NER model and training code
Phase-1 backbone + token-classification head. `bigru` adds a small bidirectional GRU over the backbone states (a causal decoder only sees its left context; the GRU supplies the right context). `linear` is a single linear layer.

In [69]:
class NERModel(nn.Module):
    def __init__(self, backbone, d_model, num_classes, head='linear', hidden=256, gru_layers=2):
        super().__init__()
        self.backbone, self.head_type, self.drop = backbone, head, nn.Dropout(0.1)
        if head == 'bigru':
            self.gru = nn.GRU(
                d_model, 
                hidden, 
                num_layers=gru_layers,                          # <--- Set number of layers here
                dropout=0.1 if gru_layers > 1 else 0.0,         # Dropout between GRU layers
                batch_first=True, 
                bidirectional=True
            )
            d_model = 2 * hidden
        self.out = nn.Linear(d_model, num_classes)

    def forward(self, ids, mask):
        with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == 'cuda')):
            h = backbone_features(self.backbone, ids, mask)
        h = self.drop(h.float())
        if self.head_type == 'bigru':
            packed = nn.utils.rnn.pack_padded_sequence(
                h, mask.sum(1).clamp(min=1).cpu(), batch_first=True, enforce_sorted=False
            )
            h, _ = nn.utils.rnn.pad_packed_sequence(
                self.gru(packed)[0], batch_first=True, total_length=ids.size(1)
            )
            h = self.drop(h)
        return self.out(h)                                          # (batch, tokens, NUM_CLASSES)


def count_params(model):
    return sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad)

def build_ner(head, method, keep_layers=None, gru_layers=2):
    """Fresh Phase-1 backbone -> adaptation method -> NER head."""
    backbone = load_phase1_backbone(keep_layers)
    apply_method(backbone, method)
    ner = NERModel(backbone, D_MODEL, NUM_CLASSES, head, gru_layers=gru_layers).to(device)
    total, trainable = count_params(ner)
    print(f'{method} | head={head} ({gru_layers} layers) | {total / 1e6:.1f}M parameters, {trainable / 1e6:.2f}M trainable ({100 * trainable / total:.1f}%)')
    return ner, dict(method=method, head=head, gru_layers=gru_layers, total_params=total, trainable_params=trainable)

In [70]:
# helper: inference on whole documents (sliding windows, overlaps blended)
@torch.no_grad()
def predict_logprobs(wrap, doc):
    """Accumulates true probabilities (avoiding log-space degradation) with edge-damped weighting."""
    wrap.eval()
    n = len(doc.get('ids', []))
    if n == 0:
        return np.zeros((0, NUM_CLASSES), dtype=np.float32)

    acc = np.zeros((n, NUM_CLASSES), dtype=np.float32)
    wsum = np.zeros(n, dtype=np.float32)
    starts = window_starts(n, WINDOW, STRIDE) or [0]

    for i in range(0, len(starts), 16):
        chunk = starts[i:i + 16]
        ids, _, mask = make_batch([doc], [(0, s) for s in chunk])
        # Accumulate probabilities in linear space:
        probs = F.softmax(wrap(ids, mask), dim=-1).cpu().numpy()
        for j, s in enumerate(chunk):
            L = min(WINDOW, n - s)
            # Ramp weight so zero-context edge tokens don't degrade predictions
            pos = np.arange(1, L + 1)
            dist_to_edge = np.minimum(pos, L - pos + 1)
            w = np.clip(dist_to_edge, 1.0, 64.0).astype(np.float32)
            acc[s:s + L] += probs[j, :L] * w[:, None]
            wsum[s:s + L] += w

    avg_prob = acc / np.maximum(wsum, 1e-9)[:, None]
    return np.log(np.maximum(avg_prob, 1e-12))


def predict_many(wrap, docs):
    return {d['id']: predict_logprobs(wrap, d) for d in docs}


def evaluate_docs(wrap, docs):
    lps = predict_many(wrap, docs)
    pred = {d['id']: decode_doc(d, lps[d['id']], 'viterbi') for d in docs}
    return strict_scores({d['id']: d['ents'] for d in docs}, pred)

### Training loop pieces: optimizer, one epoch, early stopping
Loss = class-weighted cross-entropy (plus, in the distillation notebook, a KL term to the teacher's soft labels). AdamW, linear warm-up then cosine decay, gradient clipping, fp16/bf16 autocast.

In [71]:
# helper: optimizer, trainable-weight snapshots, one training epoch
def make_optimizer(model, lr, head_lr):
    is_head = lambda n: n.startswith(('out.', 'gru.'))
    head = [p for n, p in model.named_parameters() if p.requires_grad and is_head(n)]
    body = [p for n, p in model.named_parameters() if p.requires_grad and not is_head(n)]
    groups = [dict(params=head, lr=head_lr, base_lr=head_lr)] + ([dict(params=body, lr=lr, base_lr=lr)] if body else [])
    return torch.optim.AdamW(groups, weight_decay=0.01)


def lr_factor(step, total, warmup):
    if step < warmup:
        return (step + 1) / warmup
    p = min(1.0, (step - warmup) / max(1, total - warmup))
    return 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * p))


def get_trainable_state(model):
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}


def load_trainable_state(model, state):
    params = dict(model.named_parameters())
    with torch.no_grad():
        for n, v in state.items():
            params[n].copy_(v)


def train_epoch(wrap, opt, scaler, docs, items, step, total_steps, teacher=None, kd_alpha=0.5, kd_T=2.0):
    wrap.train()
    params = [p for g in opt.param_groups for p in g['params']]
    warmup, losses = max(1, int(0.06 * total_steps)), []
    for bi in tqdm(range(0, len(items), BATCH_SIZE), leave=False):
        ids, lab, mask = make_batch(docs, items[bi:bi + BATCH_SIZE])
        logits = wrap(ids, mask)
        loss = F.cross_entropy(logits.reshape(-1, NUM_CLASSES), lab.reshape(-1), weight=CLASS_W, ignore_index=-100)
        if teacher is not None:                                              # knowledge distillation
            with torch.no_grad():
                t_logits = teacher(ids, mask)
            keep = lab.reshape(-1) != -100
            s, t = logits.reshape(-1, NUM_CLASSES)[keep], t_logits.reshape(-1, NUM_CLASSES)[keep]
            kd = F.kl_div(F.log_softmax(s / kd_T, -1), F.softmax(t / kd_T, -1), reduction='batchmean') * kd_T ** 2
            loss = kd_alpha * loss + (1 - kd_alpha) * kd
        scaler.scale(loss / GRAD_ACCUM).backward()
        losses.append(float(loss.detach()))
        if (bi // BATCH_SIZE + 1) % GRAD_ACCUM == 0 or bi + BATCH_SIZE >= len(items):
            for g in opt.param_groups:
                g['lr'] = g['base_lr'] * lr_factor(step, total_steps, warmup)
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            step += 1
            if MAX_STEPS and step >= MAX_STEPS:
                return float(np.mean(losses)), step, True
    return float(np.mean(losses)), step, False

In [72]:
def run_training(model, tag, train_docs, val_docs, *, epochs, lr, head_lr=1e-3, teacher=None, kd_alpha=0.5, kd_T=2.0):
    """Trains `model`, keeps the best epoch on validation strict macro-F1 (early stopping), restores it, returns a summary dict."""
    wrap = nn.DataParallel(model) if N_GPU > 1 else model
    opt = make_optimizer(model, lr, head_lr)
    scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda' and amp_dtype == torch.float16))
    pos, neg = make_train_items(train_docs)
    n_keep = int(NEG_KEEP * len(neg))
    steps_per_epoch = max(1, math.ceil((len(pos) + n_keep) / (BATCH_SIZE * GRAD_ACCUM)))
    total_steps = min(steps_per_epoch * epochs, MAX_STEPS or 10 ** 9)
    rng, best, history, bad, step, t0 = np.random.RandomState(SEED), dict(f1=-1.0, epoch=0, state=None), [], 0, 0, time.time()
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    print(f'[{tag}] {len(pos) + n_keep} windows per epoch, {total_steps} optimizer steps in total')
    for ep in range(1, epochs + 1):
        items = pos + ([neg[i] for i in rng.choice(len(neg), n_keep, replace=False)] if n_keep else [])
        rng.shuffle(items)
        loss, step, stop = train_epoch(wrap, opt, scaler, train_docs, items, step, total_steps, teacher, kd_alpha, kd_T)
        row = dict(epoch=ep, train_loss=round(loss, 4))
        if val_docs:
            sc = evaluate_docs(wrap, val_docs)
            row.update(val_macro_f1=round(sc['macro_f1'], 4), val_macro_p=round(sc['macro_p'], 4), val_macro_r=round(sc['macro_r'], 4))
            score = sc['macro_f1']
        else:
            score = ep                                                      # no validation: simply keep the last epoch
        improved = score > best['f1']
        if improved:
            best.update(f1=score, epoch=ep, state=get_trainable_state(model))
        bad = 0 if improved else bad + 1
        history.append(row)
        print(f'[{tag}]', row, '  <- best' if improved and val_docs else '')
        if stop or (val_docs and bad >= PATIENCE):
            break
    load_trainable_state(model, best['state'])
    peak = torch.cuda.max_memory_allocated() / 1e9 if device == 'cuda' else 0.0
    print(f'[{tag}] finished in {(time.time() - t0) / 60:.1f} min | best epoch {best["epoch"]} | best val macro-F1 {best["f1"]:.4f} | peak GPU memory {peak:.1f} GB')
    return dict(history=history, best_epoch=best['epoch'], best_f1=best['f1'], seconds=time.time() - t0, peak_mem_gb=peak)

## 11. Evaluation and submission helpers

In [73]:
# helper: compare decoding strategies on validation, then report
TAU_GRID = [0.0, 0.3, 0.4, 0.5, 0.6, 0.7]                                   # span-confidence thresholds to try


def evaluate_and_tune(model, tag, docs=None):
    """Tries greedy / viterbi x confidence thresholds on held-out documents; prints scores + error analysis; returns the best setting."""
    docs = docs if docs is not None else val_docs
    wrap = nn.DataParallel(model) if N_GPU > 1 else model
    t0 = time.time()
    lps = predict_many(wrap, docs)
    tok_per_s = sum(len(d['ids']) for d in docs) / max(time.time() - t0, 1e-9)
    gold_ = {d['id']: d['ents'] for d in docs}
    best, grid = dict(f1=-1), []
    for mode in ('greedy', 'viterbi'):
        raw = {d['id']: decode_doc(d, lps[d['id']], mode) for d in docs}
        for tau in TAU_GRID:
            pred = {k: [x for x in v if x[3] >= tau] for k, v in raw.items()}
            sc = strict_scores(gold_, pred)
            grid.append(dict(mode=mode, tau=tau, macro_f1=sc['macro_f1']))
            if sc['macro_f1'] > best['f1']:
                best = dict(f1=sc['macro_f1'], mode=mode, tau=tau, scores=sc, pred=pred)
    print(f'[{tag}] strict macro-F1 per decoding strategy:')
    for mode in ('greedy', 'viterbi'):
        print(f'   {mode:8s}', '  '.join(f'tau={r["tau"]}: {r["macro_f1"]:.4f}' for r in grid if r['mode'] == mode))
    print(f'[{tag}] best = {best["mode"]}, tau = {best["tau"]}  ({tok_per_s:.0f} tokens/s inference)')
    print_scores(best['scores'], tag)
    errors = error_analysis({d['id']: d for d in docs}, gold_, best['pred'])
    return dict(decoding=dict(mode=best['mode'], tau=best['tau']), grid=grid, scores=best['scores'],
                errors=errors, infer_tok_per_s=tok_per_s)

In [74]:
# helper: test predictions -> validated submission files
def predict_test(model, decoding):
    wrap = nn.DataParallel(model) if N_GPU > 1 else model
    lps = predict_many(wrap, test_docs)
    return {d['id']: [x for x in decode_doc(d, lps[d['id']], decoding['mode']) if x[3] >= decoding['tau']] for d in test_docs}


def to_submission(pred, docs):
    """{doc_id: [[start, end, label], ...]} in the gold offset convention, every span range-checked."""
    out = {}
    for d in docs:
        rows = []
        for s, e, lab, _ in sorted(pred[d['id']]):
            assert 0 <= s < e <= len(d['text']) and lab in LAB2ID, (d['id'], s, e, lab)
            rows.append([int(s), int(e - 1 if END_INCLUSIVE else e), lab])
        out[d['id']] = rows
    return out


def write_submission(sub, out_dir, name='prediction.json', zip_name='submission.zip'):
    path = os.path.join(out_dir, name)
    json.dump(sub, open(path, 'w', encoding='utf-8'), ensure_ascii=False)
    with zipfile.ZipFile(os.path.join(out_dir, zip_name), 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname=name)
    print(f'wrote {path} + {zip_name}: {len(sub)} documents, {sum(len(v) for v in sub.values())} entities')


def show_examples(sub, docs, n=8):
    """Prints the text under the first predicted spans so you can eyeball the offsets."""
    by_id, shown = {d['id']: d for d in docs}, 0
    for k, rows in sub.items():
        for s, e, lab in rows[:2]:
            if shown < n:
                print(f'   doc {k} [{s}:{e}] {lab:9s} {by_id[k]["text"][s:(e + 1 if END_INCLUSIVE else e)]!r}'); shown += 1
    assert set(sub) == set(by_id), 'every test document needs an entry (possibly empty)'

## 12. Run the experiment
### 12a. Build the model

In [75]:
ner, info = build_ner(HEAD, METHOD)

lorma | head=bigru (2 layers) | 154.1M parameters, 10.29M trainable (6.7%)


### 12b. Train
(If `RESUME_TRAINABLE` is set in the config, the saved weights are loaded instead and training is skipped.)

In [76]:
if RESUME_TRAINABLE:
    load_trainable_state(ner, torch.load(RESUME_TRAINABLE, map_location='cpu', weights_only=False)['trainable'])
    train_res = None
else:
    train_res = run_training(ner, RUN_NAME, train_docs, val_docs, epochs=EPOCHS, lr=LR, head_lr=HEAD_LR)

[p2_arch1_lorma] 390 windows per epoch, 625 optimizer steps in total


  0%|          | 0/49 [00:00<?, ?it/s]

[p2_arch1_lorma] {'epoch': 1, 'train_loss': 1.7126, 'val_macro_f1': 0.0175, 'val_macro_p': 0.0172, 'val_macro_r': 0.0191}   <- best


  0%|          | 0/49 [00:00<?, ?it/s]

[p2_arch1_lorma] {'epoch': 2, 'train_loss': 0.4595, 'val_macro_f1': 0.3682, 'val_macro_p': 0.3282, 'val_macro_r': 0.4508}   <- best


  0%|          | 0/49 [00:00<?, ?it/s]

... (23 outputs left out)

  0%|          | 0/49 [00:00<?, ?it/s]

[p2_arch1_lorma] {'epoch': 15, 'train_loss': 0.0089, 'val_macro_f1': 0.5999, 'val_macro_p': 0.5583, 'val_macro_r': 0.6755} 


  0%|          | 0/49 [00:00<?, ?it/s]

[p2_arch1_lorma] {'epoch': 16, 'train_loss': 0.0083, 'val_macro_f1': 0.5957, 'val_macro_p': 0.5683, 'val_macro_r': 0.648} 
[p2_arch1_lorma] finished in 18.2 min | best epoch 11 | best val macro-F1 0.6030 | peak GPU memory 8.2 GB


### 12c. Validate: scores, decoding strategies, error analysis

In [77]:
if val_docs:
    val_res = evaluate_and_tune(ner, RUN_NAME)
    decoding = val_res['decoding']
else:
    val_res, decoding = None, dict(mode='viterbi', tau=0.0)

[p2_arch1_lorma] strict macro-F1 per decoding strategy:
   greedy   tau=0.0: 0.5591  tau=0.3: 0.5598  tau=0.4: 0.5634  tau=0.5: 0.5733  tau=0.6: 0.5873  tau=0.7: 0.6124
   viterbi  tau=0.0: 0.6030  tau=0.3: 0.6035  tau=0.4: 0.6043  tau=0.5: 0.6107  tau=0.6: 0.6132  tau=0.7: 0.6225
[p2_arch1_lorma] best = viterbi, tau = 0.7  (14490 tokens/s inference)
--- p2_arch1_lorma strict macro-P 0.6118 | macro-R 0.6575 | macro-F1 0.6225 | micro-F1 0.6930
   APP        P 0.137 R 0.233 F1 0.173  (support    30, tp     7, fp    44, fn    23)
   RESP       P 0.349 R 0.625 F1 0.448  (support    24, tp    15, fp    28, fn     9)
   A.COUNSEL  P 0.733 R 0.647 F1 0.688  (support    17, tp    11, fp     4, fn     6)
   R.COUNSEL  P 0.583 R 0.500 F1 0.538  (support    14, tp     7, fp     5, fn     7)
   JUDGE      P 0.667 R 0.741 F1 0.702  (support    27, tp    20, fp    10, fn     7)
   WIT        P 0.541 R 0.323 F1 0.404  (support    62, tp    20, fp    17, fn    42)
   AUTH       P 0.488 R 0.800 F1 0.60

### 12d. Predict the test fold and write the submission

In [78]:
submission = to_submission(predict_test(ner, decoding), test_docs)
write_submission(submission, OUTPUT_DIR)
show_examples(submission, test_docs)

wrote /kaggle/working/p2_arch1_lorma/prediction.json + submission.zip: 35 documents, 4099 entities
   doc 1128835 [18:40] COURT     'SUPREME COURT OF INDIA'
   doc 1128835 [70:98] CASENO    'CIVIL APPEAL NO.6580 OF 2008'
   doc 450185 [12:22] APP       'B. SHAH Vs'
   doc 450185 [36:79] RESP      'PRESIDING OFFICER, LABOUR COURT, COIMBATORE'
   doc 13370078 [20:42] COURT     'SUPREME COURT OF INDIA'
   doc 13370078 [131:144] CASENO    'No.10247/2019'
   doc 83838189 [0:9] CASENO    'C.M.A(MD)'
   doc 83838189 [9:23] CASENO    'No.310 of 2020'


### 12e. Save weights and results

In [79]:
torch.save(dict(trainable=get_trainable_state(ner), head=HEAD, decoding=decoding, labels=LABELS, pretok=PRETOK,
                window=WINDOW, stride=STRIDE, phase1_ckpt=ckpt_path), os.path.join(OUTPUT_DIR, f'{RUN_NAME}_trainable.pt'))
json.dump(dict(run=RUN_NAME, arch=ARCH_NAME, adaptation=info, pretok=PRETOK, window=WINDOW, stride=STRIDE,
               train=train_res, validation=val_res), open(os.path.join(OUTPUT_DIR, 'results.json'), 'w'), indent=1, default=str)
print('saved:', sorted(os.listdir(OUTPUT_DIR)))

saved: ['p2_arch1_lorma_trainable.pt', 'prediction.json', 'results.json', 'submission.zip']
